# Experiment 6: Word Embeddings (GloVe) and N-gram Language Model

**Aim**
1. To implement the GloVe (Global Vectors for Word Representation) algorithm to generate word embeddings from a text corpus and find the most similar words to "king":
   - Part 1: GloVe from scratch (no pretrained model)
   - Part 2: GloVe using a pretrained model (Gensim)
2. To implement a statistical N-gram language model from scratch to predict the next word in a sequence from the preceding words (Part 3).

**Dataset**: `glove_corpus.txt` (Parts 1 and 2) and a small 8-sentence corpus given in the experiment (Part 3).

## Part 1: GloVe from Scratch (without any pretrained model)

**Theory:** GloVe learns word vectors from a global word-word co-occurrence matrix `X`, where `X[i][j]` is how often word `j` appears near word `i`.
The model learns vectors so that their dot product approximates the log co-occurrence count:

`w_i · w~_j + b_i + b~_j ≈ log(X_ij)`

The loss is a weighted least-squares: `J = Σ f(X_ij) (w_i · w~_j + b_i + b~_j − log X_ij)²`, with weighting `f(x) = (x / x_max)^α` if `x < x_max`, else `1`.

**Steps:** load corpus → build vocabulary → build co-occurrence matrix → train with gradient descent (AdaGrad) → final embedding = `W + W~` → cosine similarity to find words similar to "king".

In [1]:
import re

with open("glove_corpus.txt", "r") as f:
    sentences = [re.findall(r"[a-z]+", line.lower()) for line in f if line.strip()]

print(len(sentences), "sentences")
print(sentences[0])

24 sentences
['the', 'king', 'rules', 'the', 'kingdom']


In [2]:
vocab = sorted({w for s in sentences for w in s})
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab)
print("Vocabulary size:", V)

Vocabulary size: 68


In [3]:
import numpy as np

window = 2
X = np.zeros((V, V))

for s in sentences:
    for i, w in enumerate(s):
        for j in range(max(0, i - window), min(len(s), i + window + 1)):
            if i != j:
                X[word2idx[w], word2idx[s[j]]] += 1.0 / abs(i - j)

print("Non-zero entries:", np.count_nonzero(X))

Non-zero entries: 342


In [4]:
dim = 10
x_max, alpha, lr, epochs = 10, 0.75, 0.05, 500

rng = np.random.default_rng(42)
W  = rng.uniform(-0.5, 0.5, (V, dim)) / dim
Wc = rng.uniform(-0.5, 0.5, (V, dim)) / dim
b, bc = np.zeros(V), np.zeros(V)

# AdaGrad accumulators
gW, gWc = np.ones((V, dim)), np.ones((V, dim))
gb, gbc = np.ones(V), np.ones(V)

pairs = np.argwhere(X > 0)

for epoch in range(epochs):
    rng.shuffle(pairs)
    loss = 0.0
    for i, j in pairs:
        x = X[i, j]
        f = (x / x_max) ** alpha if x < x_max else 1.0
        diff = W[i] @ Wc[j] + b[i] + bc[j] - np.log(x)
        loss += 0.5 * f * diff ** 2

        grad = f * diff
        dW, dWc = grad * Wc[j], grad * W[i]

        W[i]  -= lr * dW  / np.sqrt(gW[i])
        Wc[j] -= lr * dWc / np.sqrt(gWc[j])
        b[i]  -= lr * grad / np.sqrt(gb[i])
        bc[j] -= lr * grad / np.sqrt(gbc[j])

        gW[i] += dW ** 2
        gWc[j] += dWc ** 2
        gb[i] += grad ** 2
        gbc[j] += grad ** 2

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}, loss = {loss:.4f}")

Epoch 100, loss = 2.0373
Epoch 200, loss = 0.7764
Epoch 300, loss = 0.3507
Epoch 400, loss = 0.1178
Epoch 500, loss = 0.0400


In [5]:
embeddings = W + Wc

def most_similar(word, topn=5):
    v = embeddings[word2idx[word]]
    sims = embeddings @ v / (np.linalg.norm(embeddings, axis=1) * np.linalg.norm(v))
    order = np.argsort(-sims)
    return [(idx2word[i], round(float(sims[i]), 4)) for i in order if idx2word[i] != word][:topn]

for w, s in most_similar("king", 5):
    print(f"{w:12s} {s}")

queen        0.9841
kingdom      0.9636
rules        0.856
royal        0.8512
night        0.8486


## Part 2: GloVe using a Pretrained Model (Gensim)

**Theory:** Instead of training on our tiny corpus, we load GloVe vectors that were already trained on a very large corpus (Wikipedia + Gigaword, 6B tokens) using Gensim's downloader. Because they were trained on huge data, the similar words are much more meaningful.

**Steps:** install/import Gensim → load pretrained `glove-wiki-gigaword-100` → find the most similar words to "king".

In [6]:
!pip install -q gensim

In [7]:
import gensim.downloader as api

glove_model = api.load("glove-wiki-gigaword-100")
print("Vocabulary size:", len(glove_model.key_to_index))

[==================================================] 100.0% 128.1/128.1MB downloaded
Vocabulary size: 400000


In [8]:
for w, s in glove_model.most_similar("king", topn=5):
    print(f"{w:12s} {s:.4f}")

prince       0.7682
queen        0.7508
son          0.7021
brother      0.6986
monarch      0.6978


## Part 3: N-gram Language Model from Scratch (Next Word Prediction)

**Theory:** An N-gram model estimates the probability of the next word from the previous `N-1` words using counts:

`P(w_n | w_1 ... w_{n-1}) = count(w_1 ... w_n) / count(w_1 ... w_{n-1})`

A **trigram** model (N = 3) is built here. If the two-word context was never seen, it **backs off** to a bigram model (one word of context).

**Steps:** define corpus → tokenize → count bigrams and trigrams → compute probabilities → predict the next word.

In [9]:
corpus = """natural language processing is a branch of artificial intelligence.
natural language processing enables computers to understand human language.
machine learning is a branch of artificial intelligence.
deep learning is a part of machine learning.
language models learn patterns from text data.
a language model predicts the next word in a sentence.
natural language processing uses language models.
artificial intelligence includes machine learning and deep learning."""

tokenized = [line.replace(".", "").split() for line in corpus.split("\n")]
print(tokenized[0])

['natural', 'language', 'processing', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence']


In [10]:
from collections import defaultdict, Counter

bigram_counts = defaultdict(Counter)    # word -> Counter(next words)
trigram_counts = defaultdict(Counter)   # (w1, w2) -> Counter(next words)

for words in tokenized:
    for i in range(len(words) - 1):
        bigram_counts[words[i]][words[i + 1]] += 1
    for i in range(len(words) - 2):
        trigram_counts[(words[i], words[i + 1])][words[i + 2]] += 1

In [11]:
def predict_next(text, topn=3):
    words = text.lower().split()
    if len(words) >= 2 and tuple(words[-2:]) in trigram_counts:
        counts = trigram_counts[tuple(words[-2:])]
    elif words[-1] in bigram_counts:
        counts = bigram_counts[words[-1]]
    else:
        return []
    total = sum(counts.values())
    return [(w, round(c / total, 3)) for w, c in counts.most_common(topn)]

In [12]:
tests = ["natural language", "language", "machine learning", "is a", "artificial intelligence", "a language"]

for t in tests:
    print(f"'{t}' -> {predict_next(t)}")

'natural language' -> [('processing', 1.0)]
'language' -> [('processing', 0.5), ('models', 0.333), ('model', 0.167)]
'machine learning' -> [('is', 0.5), ('and', 0.5)]
'is a' -> [('branch', 0.667), ('part', 0.333)]
'artificial intelligence' -> [('includes', 1.0)]
'a language' -> [('model', 1.0)]


## Conclusion
- GloVe from scratch learned embeddings from the small corpus using co-occurrence counts; words appearing in similar contexts to "king" (such as "queen") get higher similarity.
- The pretrained Gensim GloVe model gives far better similar words because it was trained on billions of tokens.
- The N-gram model predicted the next word from counts of previous words and gave probabilities for each candidate.